# Tiền xử lý dữ liệu
Chuẩn bị X, target Regression/Classification và hai bộ tiền xử lý để dùng trong pipeline mô hình. Chỉ đọc Train, không xử lý lại cleaning và không fit encoding/scaling tại đây.

In [1]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "src":
    ROOT = ROOT.parent

train = pd.read_csv(ROOT / "data" / "medical_insurance_train.csv")
print(f"Train: {len(train):,} dòng, {train.shape[1]} cột")

Train: 1,069 dòng, 7 cột


## 1. Tách đặc trưng và target Regression
X gồm 6 thuộc tính đầu vào; `charges` chỉ dùng làm target, không đưa vào bộ tiền xử lý đặc trưng.

In [2]:
numeric = ["age", "bmi", "children"]
categorical = ["sex", "smoker", "region"]

X_train = train[numeric + categorical].copy()
y_train_reg = train["charges"].copy()

print("X_train:", X_train.shape)
print("y_train_reg:", y_train_reg.shape)
display(X_train.head())

X_train: (1069, 6)
y_train_reg: (1069,)


,age,bmi,children,sex,smoker,region
0,23,24.510,0,male,no,northeast
1,21,25.745,2,male,no,northeast
2,52,37.525,2,female,no,northwest
3,63,41.470,0,male,no,southeast
4,47,26.600,2,female,no,northeast


## 2. Thiết lập encoding và scaling
- One-Hot Encoding cho `sex`, `smoker`, `region`; nhãn chưa gặp khi fit được xử lý bằng `handle_unknown="ignore"`.
- Bộ **scaled**: StandardScaler cho cột số, phù hợp mô hình nhạy với thang đo như Ridge, Logistic Regression hoặc SVM.
- Bộ **tree**: giữ nguyên cột số, dành cho các mô hình cây như Random Forest/XGBoost.

Chỉ khởi tạo, chưa gọi `fit` hay `fit_transform`. Khi thử mô hình, ghép bộ phù hợp vào Pipeline để mỗi fold CV tự fit từ phần train của fold. Dùng one-hot đầy đủ; nếu thử Linear Regression không điều chuẩn, cần xét tính phụ thuộc giữa các cột dummy khi diễn giải hệ số.

In [3]:
preprocessor_scaled = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])

preprocessor_tree = ColumnTransformer([
    ("numeric", "passthrough", numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])

print("Đã tạo hai bộ tiền xử lý: scaled và tree (chưa fit).")

Đã tạo hai bộ tiền xử lý: scaled và tree (chưa fit).


## 3. Tạo target Classification
Mặc định thử nghiệm: hai ngưỡng là phân vị **1/3 và 2/3 của `charges` trên Train**. Ba mức mô tả chi phí tương đối trong dữ liệu, không phải ngưỡng chuyên môn y tế hay mức rủi ro lâm sàng.

Quy ước: **0 = Thấp**, **1 = Trung bình**, **2 = Cao**. Lưu và cố định ngưỡng để các mô hình Classification dùng cùng định nghĩa nhãn; không tính lại ngưỡng từ Test. CV đánh giá bài toán phân loại với định nghĩa nhãn này, không đánh giá việc chọn ngưỡng.

In [4]:
lower, upper = y_train_reg.quantile([1 / 3, 2 / 3]).tolist()
labels = {0: "Thấp", 1: "Trung bình", 2: "Cao"}

# right=True: ngưỡng trên thuộc về khoảng bên trái.
y_train_cls = pd.cut(
    y_train_reg,
    bins=[-float("inf"), lower, upper, float("inf")],
    labels=[0, 1, 2],
    right=True,
).astype("int64").rename("cost_level")

print(f"Thấp: charges <= {lower:,.4f} USD")
print(f"Trung bình: {lower:,.4f} < charges <= {upper:,.4f} USD")
print(f"Cao: charges > {upper:,.4f} USD")
display(pd.DataFrame({
    "Mức chi phí": pd.Series(labels),
    "Số bản ghi Train": y_train_cls.value_counts().reindex([0, 1, 2], fill_value=0),
}))

Thấp: charges <= 6,250.4350 USD
Trung bình: 6,250.4350 < charges <= 12,730.9996 USD
Cao: charges > 12,730.9996 USD


,Mức chi phí,Số bản ghi Train
0,Thấp,357
1,Trung bình,356
2,Cao,356


## 4. Lưu để các notebook thực nghiệm dùng chung
Lưu hai bộ tiền xử lý **chưa fit**, nhóm đặc trưng và ngưỡng phân loại trong một file Joblib. Không xuất CSV đã encoding/scaling toàn bộ Train. File này là cấu hình tiền xử lý, chưa phải mô hình dự đoán.

In [5]:
MODELS = ROOT / "models"
MODELS.mkdir(parents=True, exist_ok=True)
OUTPUT = MODELS / "preprocessing_setup.joblib"

setup = {
    "numeric_features": numeric,
    "categorical_features": categorical,
    "target": "charges",
    "scaled": preprocessor_scaled,
    "tree": preprocessor_tree,
    "classification_thresholds": [lower, upper],
    "classification_labels": labels,
}
joblib.dump(setup, OUTPUT)
print(f"Đã lưu: {OUTPUT.relative_to(ROOT)}")

Đã lưu: models/preprocessing_setup.joblib


**Bước tiếp theo:** các notebook thực nghiệm tải `preprocessing_setup.joblib`, đọc lại Train và ghép bộ tiền xử lý với từng mô hình. Ví dụ cấu trúc (biến `model` là mô hình đang thử):

```python
from sklearn.base import clone
from sklearn.pipeline import Pipeline

setup = joblib.load(ROOT / "models" / "preprocessing_setup.joblib")
pipeline = Pipeline([
    ("preprocessing", clone(setup["scaled"])),  # Hoặc setup["tree"]
    ("model", model),
])
```

Với Classification, tạo nhãn từ `charges` bằng đúng ngưỡng đã lưu. Dùng CV/tuning trên Train để chọn mô hình. Sau khi chốt, fit pipeline trên Train rồi đánh giá Test; nhãn Test dùng cùng ngưỡng, không tính lại phân vị. Lưu **pipeline đã huấn luyện** riêng để Streamlit tải và dự đoán.